# Preprocessing untuk Analisis Sentimen Teks Finansial Slang



Untuk menjaga agar istilah-istilah penting dalam dunia keuangan/saham tidak terdistorsi selama pembersihan dan normalisasi, kami menerapkan teknik Selective Normalization.

### 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import re
from typing import Set, Dict, Optional
import warnings
import emoji

### 2. Definisi Istilah Finansial yang Dilindungi

Mendefinisikan daftar istilah keuangan dan pasar saham penting (seperti *cuan, hold, cut loss, ara, arb, ipo, support, resist, emiten, fomo*). Istilah-istilah ini dilindungi agar tidak diterjemahkan atau diubah oleh kamus normalisasi slang umum.

In [ ]:
PROTECTED_FINANCIAL_TERMS: Set[str] = {
    #slang umum
    "cuan", "loss", "cut loss", "cutloss", "cl", "tp", "take profit",
    "hold", "buy", "sell", "entry", "exit", "average down", "avg down", "avgdown",
    "serok", "serokin", "nyerok", "nyangkut", "pompom", "pom pom",
    "mantul", "gaskeun", "gas", "gaspol", "rebound", "rally",
    "all in", "allin", "top up", "topup", "nambah", "nambahin",
    "cuci", "cuci rugi", "cuci tangan", "lepas", "lepasin",
    "tunggu", "nunggu", "hold tight", "hold dulu", "wd", "withdraw", "depo", "deposit", "porto", "portofolio",
    "floating", "fl", "floating loss", "floating profit", "fp",
    "boncos", "jebol", "terbang", "meroket", "melejit", "ambles",
    "nyungsep", "anjlok", "rebound", "rally", "koreksi",
    "konsolidasi", "akumulasi", "distribusi", "markup", "markdown",
    "bandar", "bandar gede", "bandar kecil", "bandar ritel",
    "smart money", "dumb money", "retail", "institusi",
    "whale", "ikan kecil", "ikan besar", "ikan asin",
    "cuci gudang", "cuci rugi", "cuci tangan", "lepas rugi",
    "nahan", "tahan", "diam", "diem", "diam dulu",
    "tunggu koreksi", "tunggu breakup", "tunggu breakdown",

    #mekanisme pasar
    "ara", "arb", "haka", "haki", "auto reject atas", "auto reject bawah",
    "ihsg", "idx", "bid", "offer", "lot", "spread", "volume", "turnover",
    "ipo", "right issue", "ri", "stock split", "reverse split",
    "closing", "opening", "pre-opening", "pre-closing",
    "suspensi", "delisting", "relisting", "auto reject", "ar",
    "fr", "full room", "lower auto reject", "upper auto reject",

    #istilah teknikal
    "support", "resist", "resisten", "resistance", "breakout", "breakdown",
    "bullish", "bearish", "sideways", "uptrend", "downtrend", "trend",
    "macd", "rsi", "ma", "ema", "sma", "bollinger", "fibonacci",
    "golden cross", "death cross", "volume", "bandarmology",
    "candle", "candlestick", "doji", "hammer", "shooting star",
    "double top", "double bottom", "head and shoulder", "triangle",
    "flag", "pennant", "channel", "wedge", "gap", "gap up", "gap down",
    "overbought", "oversold", "divergence", "convergence",

    #istilah fundamental
    "emiten", "saham", "dividen", "dividend", "eps", "per", "pbv",
    "roe", "roa", "der", "bv", "nav", "aum", "laba", "rugi",
    "laporan keuangan", "lapkeu", "q1", "q2", "q3", "q4",
    "full year", "fy", "revenue", "net profit", "gross profit",
    "ebitda", "ebit", "cash flow", "free cash flow", "fcf",
    "book value", "market cap", "market capitalization",
    "debt to equity", "debt to asset", "current ratio",
    "bluechip", "blue chip", "second liner", "third liner",
    "gorengan", "lq45", "idx30", "idx80", "kompas100",
    "warrant", "waran", "right", "konversi", "konversi obligasi",

    #istilah ekonoi
    "corporate action", "ca", "stock split", "reverse split",
    "right issue", "private placement", "penawaran umum",
    "penawaran umum terbatas", "put", "call", "merger", "akuisisi",
    "spin off", "divestasi", "buyback", "stock buyback",
    "dividen tunai", "dividen saham", "stock dividend",
    "agm", "egm", "rups", "rupslb", "public expose",

    #istilah psikologi trading
    "fomo", "fud", "greed", "fear", "panic selling", "panic buy",
    "overtrading", "overconfidence", "revenge trading",
    "diamond hand", "paper hand", "hodl", "hodler",
    "bag holder", "bagholder", "nyangkut", "terjebak",

    #istilah order dan eksekusi
    "market order", "limit order", "stop loss", "stop buy",
    "trailing stop", "oco", "one cancels other",
    " iceberg order", "hidden order", "best price",
}
_PROTECTED_LOWER: Set[str] = {t.lower() for t in PROTECTED_FINANCIAL_TERMS}

### 3. Normalisasi Slang Umum

Mendefinisikan pemetaan dari kata slang atau singkatan umum di media sosial Indonesia (seperti *yg, utk, dgn, dlm, tdk, gak, gw, lu, mantap*) ke bentuk bakunya (seperti *yang, untuk, dengan, dalam, tidak, tidak, saya, kamu, mantap*), serta menghapus tawa (*wkwk, haha*).

In [ ]:
NORMALIZATION_DICT: Dict[str, str] = {
    "yg": "yang", "utk": "untuk", "dgn": "dengan", "dlm": "dalam",
    "dr": "dari", "pd": "pada", "krn": "karena", "krna": "karena",
    "blm": "belum", "sdh": "sudah", "udh": "sudah", "udah": "sudah",
    "tdk": "tidak", "gak": "tidak", "ga": "tidak", "gk": "tidak",
    "ngga": "tidak", "nggak": "tidak", "engga": "tidak", "enggak": "tidak",
    "tpi": "tapi", "ttg": "tentang", "ttng": "tentang",
    "sm": "sama", "bgt": "banget", "bngt": "banget", "bngtt": "banget",
    "jg": "juga", "jga": "juga", "jgn": "jangan",
    "lg": "lagi", "lgi": "lagi", "bs": "bisa", "bsa": "bisa",
    "org": "orang", "orng": "orang", "kl": "kalau", "klo": "kalau", "kalo": "kalau",
    "gmn": "gimana", "gmna": "gimana", "emg": "emang", "emng": "emang",
    "aja": "saja", "aj": "saja", "bkn": "bukan", "bukn": "bukan",
    "skrg": "sekarang", "skrng": "sekarang", "hr": "hari", "hri": "hari",
    "msh": "masih", "msih": "masih", "sy": "saya", "ak": "aku",
    "gw": "saya", "gue": "saya", "gua": "saya",
    "lu": "kamu", "lo": "kamu", "elu": "kamu",
    "mantap": "mantap", "mantab": "mantap", "mantep": "mantap",
    "bagus": "bagus", "bgus": "bagus", "jelek": "jelek", "jlek": "jelek",
    "wkwk": "", "wkwkwk": "", "haha": "", "hahaha": "",
    "btw": "ngomong-ngomong", "otw": "sedang menuju",
}
NORMALIZATION_DICT = {k: v for k, v in NORMALIZATION_DICT.items() if v is not None}

### 4. Pembersihan Teks awal

Disini kami membuat fungsi `clean_text` untuk:
- Menghapus bracket/placeholder seperti `[USERNAME]`, `[URL]`, `[HASHTAG]`.
- Menghapus link HTTP/HTTPS dan mention `@username`.
- Menghapus emoji menggunakan pustaka `emoji`.
- Menghapus simbol hashtag `#` tapi mempertahankan katanya.
- Mempertahankan ticker saham dengan membuang simbol `$` (misal: `$BBCA` $\rightarrow$ `BBCA`).
- Menghapus karakter khusus, angka berdiri sendiri, line breaks (`\n`), dan spasi berlebih.
- Mengurangi pengulangan karakter berlebih (misal: *yaaa* menjadi *yaa*).

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str) or text.strip() == "":
        return ""
    #Placeholders: [USERNAME], [URL], [HASHTAG], apapun dalam bracket
    text = re.sub(r"\[[^\]]*\]", "", text)
    #URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    #Mentions
    text = re.sub(r"@\w+", "", text)
    #Emoji
    text = emoji.replace_emoji(text, replace="")
    #Hashtag tapi tetep pertahankan kata
    text = re.sub(r"#(\w+)", r"\1", text)
    #$BBCA jadi BBCA (pertahankan nama saham)
    text = re.sub(r"\$([A-Za-z]+)", r"\1", text)
    #Superscript dan simbol khusus
    text = re.sub(r"[²³¹⁰⁴⁵⁶⁷⁸⁹™©®°]", "", text)
    #Angka berdiri sendiri
    text = re.sub(r"(?<![A-Za-z])\d+(?![A-Za-z])", "", text)
    #Karakter non-alfanumerik
    text = re.sub(r"[^\w\s\-]", " ", text)
    #Line breaks
    text = re.sub(r"[\n\r\\n]", " ", text)
    #Kata berulang kayak yaaa -> yaa, ehehehe -> ehe, wkwkwk -> wkwk
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    #Whitespace normalization
    text = re.sub(r"\s+", " ", text).strip()
    return text


### 5. Fungsi Pengecekan Istilah yang Dilindungi

Fungsi pembantu `is_protected` untuk mendeteksi apakah suatu token termasuk dalam daftar istilah finansial yang dilindungi.

In [ ]:
def is_protected(token: str) -> bool:
    return token.lower() in _PROTECTED_LOWER

### 6. Fungsi Normalisasi Token Tunggal

Fungsi `normalize_token` mengonversi token menjadi huruf kecil. Jika kata tersebut tergolong dalam daftar yang dilindungi, ia akan dikembalikan apa adanya. Jika ada di kamus slang umum, ia akan dikonversi ke bentuk baku. Jika tidak keduanya, ia dikembalikan sebagai huruf kecil biasa.

In [ ]:
def normalize_token(token: str) -> str:
    lower = token.lower()
    if lower in _PROTECTED_LOWER:
        return lower
    if lower in NORMALIZATION_DICT:
        return NORMALIZATION_DICT[lower]
    return lower


### 7. Selective Normalization

Fungsi `selective_normalize` memecah teks menjadi token-token dan mengeceknya secara bertahap:
- Memeriksa kombinasi dua kata (bigram) untuk mendeteksi istilah dilindungi dua kata (seperti *cut loss*, *auto reject*).
- Jika terdeteksi, bigram tersebut digabungkan dan dilewati.
- Jika tidak, dilakukan normalisasi token tunggal.

In [ ]:
def selective_normalize(text: str) -> str:
    tokens = text.split()
    result = []
    i = 0
    while i < len(tokens):
        if i + 1 < len(tokens):
            bigram = f"{tokens[i].lower()} {tokens[i+1].lower()}"
            if bigram in _PROTECTED_LOWER:
                result.append(bigram)
                i += 2
                continue
        normalized = normalize_token(tokens[i])
        if normalized:
            result.append(normalized)
        i += 1
    return " ".join(result)


### 8. Fungsi Utama Preprocessing Teks

Fungsi `preprocess_text` menggabungkan langkah pembersihan teks dasar (`clean_text`) dan normalisasi selektif (`selective_normalize`) menjadi satu alur pipeline untuk setiap string teks.

In [ ]:
def preprocess_text(text: str) -> str:

    text = clean_text(text)
    text = selective_normalize(text)
    return text

### 9. Pipeline Utama Pemrosesan Dataset (*Main Pipeline*)

Fungsi `main` memuat dataset gabungan (`dataset_combined_final.csv`), memvalidasi kolom input (`text` dan `sentiment`), menjalankan pembersihan teks pada seluruh baris, menghapus baris kosong/invalid setelah pembersihan, menampilkan statistik distribusi sentimen dan sumber data, menampilkan contoh hasil sebelum-sesudah preprocessing, serta menyimpan dataset bersih ke `dataset_final_clean.csv`.

In [ ]:
def main():
    try:
        df = pd.read_csv('dataset_combined_final.csv')
        print(f"Loaded: {len(df)} records × {df.shape[1]} columns")
        print(f"Columns: {list(df.columns)}")
    except FileNotFoundError:
        print("ERROR")
        return None
    # Step 2: Validate input
    if 'text' not in df.columns:
        print("ERROR")
        return None
    if 'sentiment' not in df.columns:
        print("ERROR")
        return None
    print(f"Columns oke")
    print(f"Missing values before cleaning: {df['text'].isna().sum() + df['sentiment'].isna().sum()}")
    # Step 3: Apply preprocessing
    print("\nApplying preprocessing...")
    print("Cleaning (URL, mentions, emoji, line breaks, etc)")
    print("Selective normalization (slang → formal, protected financial terms)")
    df['text_cleaned'] = df['text'].apply(preprocess_text)
    print("done")


    initial_count = len(df)
    # Drop rows dengan text kosong setelah preprocessing
    df = df[df['text_cleaned'].str.len() > 0]
    # Drop rows dengan sentiment kosong
    df = df[df['sentiment'].notna()]
    removed_count = initial_count - len(df)
    print(f"Removed {removed_count} invalid rows")
    print(f"Remaining: {len(df)} records")
    #Reorder dan select columns
    final_df = df[['text', 'text_cleaned', 'sentiment', 'source']].copy()
    final_df.columns = ['text_original', 'text_cleaned', 'sentiment', 'source']
    #analusis
    sentiment_dist = final_df['sentiment'].value_counts()
    for label in ['Positive', 'Negative', 'Neutral']:
        count = sentiment_dist.get(label, 0)
        prop = (count / len(final_df) * 100) if len(final_df) > 0 else 0
        print(f"   {label:12} : {count:5} ({prop:5.1f}%)")
    print("\nSource Distribution:")
    source_dist = final_df['source'].value_counts()
    for source, count in source_dist.items():
        prop = (count / len(final_df) * 100) if len(final_df) > 0 else 0
        print(f"{source:30} : {count:5} ({prop:5.1f}%)")
    print("\nText Length Statistics (after cleaning):")
    text_lens = final_df['text_cleaned'].str.len()
    print(f"Min     : {text_lens.min()}")
    print(f"Max     : {text_lens.max()}")
    print(f"Mean    : {text_lens.mean():.1f}")
    print(f"Median  : {text_lens.median():.1f}")
    print("\nSample Preprocessed Texts:")
    for idx, row in final_df.sample(min(5, len(final_df))).iterrows():
        orig = row['text_original'][:50]
        clean = row['text_cleaned'][:50]
        print(f"\nOriginal: {orig}...")
        print(f"Cleaned : {clean}...")
    #Save
    output_path = 'dataset_final_clean.csv'
    final_df.to_csv(output_path, index=False)
    
    print(f"   Records: {len(final_df)}")
    print(f"   Columns: {list(final_df.columns)}")
    print(f"   File size: {final_df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
    return final_df

### 10. Eksekusi Preprocessing Pipeline

Menjalankan fungsi `main()` untuk memproses seluruh dataset dan menyimpan hasilnya.

In [ ]:
main()

Loaded: 6140 records × 3 columns
Columns: ['text', 'sentiment', 'source']
Columns oke
Missing values before cleaning: 13

Applying preprocessing...
Cleaning (URL, mentions, emoji, line breaks, etc)
Selective normalization (slang → formal, protected financial terms)
done
Removed 16 invalid rows
Remaining: 6124 records
   Positive     :  2851 ( 46.6%)
   Negative     :  1445 ( 23.6%)
   Neutral      :  1828 ( 29.8%)

Source Distribution:
Twitter (IDSMSA)               :  3288 ( 53.7%)
Stockbit (ICCSCI)              :  1842 ( 30.1%)
Stockbit (ARTO)                :   497 (  8.1%)
Stockbit (PTBA)                :   497 (  8.1%)

Text Length Statistics (after cleaning):
Min     : 2
Max     : 12216
Mean    : 138.5
Median  : 76.0

Sample Preprocessed Texts:

Original: IHSG Diprediksi Menguat, Saham ASII hingga INDF Di...
Cleaned : ihsg diprediksi me guat saham asii hi gga indf dir...

Original: BANK ayoo beli dan hold keras semuanyaa jangan kas...
Cleaned : bank ayoo beli da hold keras semua 

: 

### 11. Verifikasi Hasil Preprocessing

Membaca dataset bersih hasil pembersihan (`dataset_final_clean.csv`) dan menampilkan beberapa sampel data teratas untuk memastikan kolom `text_original` dan `text_cleaned` sudah terbuat dengan benar.

In [3]:
import pandas as pd
df = pd.read_csv(r'C:\BINUS\NLP\Research_paper\CS-IDX30 version 3\Dataset\dataset_final_clean.csv')
df.head(-1)

,text_original,text_cleaned,sentiment,source
0,"Gk muluk muluk, 100,000 lot saham BBCA aja",tidak muluk muluk 100 00 lot saham bbca saja,Positive,Twitter (IDSMSA)
1,BCA Expoversary 2024 menawarkan promo suku bun...,bca expoversary 2024 menawarkan promo suku bun...,Neutral,Twitter (IDSMSA)
2,[USERNAME] saham bca nya menyusul ya 🙂,saham bca nya menyusul ya,Positive,Twitter (IDSMSA)
3,PT Bank BCA Syariah (BCA Syariah) turut memeri...,pt bank bca syariah bca syariah turut memeriah...,Neutral,Twitter (IDSMSA)
4,[USERNAME] Begitu byk saham kamu memilih saham...,begitu banyak saham kamu memilih saham itu kal...,Positive,Twitter (IDSMSA)
...,...,...,...,...
6114,@robertgunawankeren duduk manis sambil menikma...,duduk manis sambil menikmati dividen itmg ptba...,Positive,Stockbit (PTBA)
6115,$MEDC Info A1 Senin bakal dikerek ke 900\n\n$I...,medc info a1 senin bakal dikerek ke 900 ihsg p...,Neutral,Stockbit (PTBA)
6116,@lukas1991 iya pak! huahaha\n\nmudah2an sopirn...,iya pak huahaha mudah2an sopirnya si bumi tida...,Neutral,Stockbit (PTBA)
6117,@lukas1991 iya pak!! huahaha\n\nsopirnya dikas...,iya pak huahaha sopirnya dikasih target akhir ...,Neutral,Stockbit (PTBA)


### 12. Analisis Distribusi Kelas Sentimen

Menampilkan jumlah data untuk masing-masing kelas sentimen (`Positive`, `Neutral`, `Negative`) untuk memeriksa keseimbangan dataset.

In [6]:
print(df['sentiment'].value_counts())

sentiment
Positive    2850
Neutral     1825
Negative    1445
Name: count, dtype: int64


### 13. Analisis Distribusi Sumber Data

Menampilkan jumlah data untuk masing-masing sumber asal data (`Twitter`, `Stockbit`) untuk mengetahui kontribusi setiap sumber terhadap dataset.

In [7]:
print(df['source'].value_counts())

source
Twitter (IDSMSA)     3288
Stockbit (ICCSCI)    1838
Stockbit (ARTO)       497
Stockbit (PTBA)       497
Name: count, dtype: int64
